In [2]:
import pandas as pd

In [3]:
import os

# Path to the datasets folder
datasets_folder = 'datasets'

# List all files in the datasets folder
files = os.listdir(datasets_folder)

# Initialize a report dictionary
report = {}

# Loop through each file and read the dataset
for file in files:
    file_path = os.path.join(datasets_folder, file)
    if file.endswith('.csv'):
        df = pd.read_csv(file_path)
        if 'Class' in df.columns:
            class_column = 'Class'
        elif 'class' in df.columns:
            class_column = 'class'
        elif 'game' in df.columns:
            class_column = 'game'
        elif 'V11' in df.columns:
            class_column = 'V11'
        else:
            class_column = None

        if class_column:
            report[file] = {'rows': df.shape[0], 'columns': df.shape[1], 'classes': len(df[class_column].unique())}
        else:
            report[file] = {'rows': df.shape[0], 'columns': df.shape[1], 'classes': 'N/A'}

# Remove entries with 'N/A' classes from the report
filtered_report = {k: v for k, v in report.items() if v['classes'] != 'N/A'}

# Sort the filtered report by number of classes, then by number of rows, and then by number of columns
sorted_report = dict(sorted(filtered_report.items(), key=lambda item: (item[1]['classes'], item[1]['rows'], item[1]['columns'])))

# Print the sorted report
for file, info in sorted_report.items():
    print(f"Dataset: {file}, Rows: {info['rows']}, Columns: {info['columns']}, Classes: {info['classes']}")

Dataset: Nursery.csv, Rows: 12960, Columns: 9, Classes: 5
Dataset: Phishing URL.csv, Rows: 18982, Columns: 80, Classes: 5
Dataset: Satimage.csv, Rows: 6430, Columns: 37, Classes: 6
Dataset: HAR.csv, Rows: 10299, Columns: 562, Classes: 6
Dataset: Mosquitoes.csv, Rows: 158249, Columns: 54, Classes: 6
Dataset: Dermatology.csv, Rows: 1000000, Columns: 35, Classes: 6
Dataset: Covertype.csv, Rows: 110393, Columns: 55, Classes: 7
Dataset: Land-use.csv, Rows: 9144, Columns: 221, Classes: 8
Dataset: Mfeat.csv, Rows: 2000, Columns: 7, Classes: 10
Dataset: Avila.csv, Rows: 20867, Columns: 11, Classes: 12
Dataset: Chess game.csv, Rows: 28056, Columns: 7, Classes: 18
Dataset: Walking.csv, Rows: 149332, Columns: 5, Classes: 22


In [4]:
dataset = "datasets/Nursery.csv"
def print_bad_lines(line):
    print(f"Bad line: {line}")

df = pd.read_csv(dataset, on_bad_lines=print_bad_lines, engine='python')
df

,parents,has_nurs,form,children,housing,finance,social,health,class
0,usual,proper,complete,1,convenient,convenient,nonprob,recommended,recommend
1,usual,proper,complete,1,convenient,convenient,nonprob,priority,priority
2,usual,proper,complete,1,convenient,convenient,nonprob,not_recom,not_recom
3,usual,proper,complete,1,convenient,convenient,slightly_prob,recommended,recommend
4,usual,proper,complete,1,convenient,convenient,slightly_prob,priority,priority
...,...,...,...,...,...,...,...,...,...
12955,great_pret,very_crit,foster,more,critical,inconv,slightly_prob,priority,spec_prior
12956,great_pret,very_crit,foster,more,critical,inconv,slightly_prob,not_recom,not_recom
12957,great_pret,very_crit,foster,more,critical,inconv,problematic,recommended,spec_prior
12958,great_pret,very_crit,foster,more,critical,inconv,problematic,priority,spec_prior


In [5]:
df['class'].unique()

array(['recommend', 'priority', 'not_recom', 'very_recom', 'spec_prior'],
      dtype=object)

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score
from sklearn.metrics import roc_auc_score

# Encode categorical variables
label_encoders = {}
for column in df.columns:
    if df[column].dtype == 'object':
        le = LabelEncoder()
        df[column] = le.fit_transform(df[column])
        label_encoders[column] = le

# Split the data into features and target
X = df.drop('class', axis=1)
y = df['class']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train the regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)
# Evaluate the model using AUC metric
auc = roc_auc_score(y_test, model.predict_proba(X_test), multi_class='ovr')
print(f'AUC: {auc}')

ValueError: Number of classes in y_true not equal to the number of columns in 'y_score'